# 第 56 章：Capstone 学期收尾与下一届 Warm Start

这个 notebook 用一个极小的 shutdown-warmstart table，对比“只按收尾压力排序”的 naive baseline 和“先检查 owner、shutdown check、archive snapshot 与 next-cohort seed”的学期收尾 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_shutdown_warmstart_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['release_pressure_score'] = float(row['release_pressure_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} shutdown items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Carry-forward scope:', ordered_counts(rows, 'carry_forward_scope'))
print('Shutdown check:', ordered_counts(rows, 'shutdown_check_status'))
print('Archive snapshot:', ordered_counts(rows, 'archive_snapshot_status'))
print('Next-cohort seed:', ordered_counts(rows, 'next_cohort_seed_status'))
print('Owner status:', ordered_counts(rows, 'owner_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['shutdown_id']
    for row in rows
    if row['reference_route'] == 'shutdown_ready'
)
top4_pressure = sorted(
    rows,
    key=lambda row: row['release_pressure_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'shutdown_ready'
    for row in top4_pressure
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'shutdown_ready'
    for row in top4_pressure
)
baseline_ready_precision = baseline_ready_hits / len(top4_pressure)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_pressure)

print('Reference shutdown-ready items:', reference_ready_ids)
print('Top-4 by release pressure:')
for row in top4_pressure:
    print(
        f"  {row['shutdown_id']}: pressure={row['release_pressure_score']:.2f}, "
        f"route={row['reference_route']}, area={row['shutdown_area']}"
    )
print('Baseline shutdown precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_shutdown_item(row):
    if row['owner_status'] != 'assigned':
        return 'assign_shutdown_owner', 'shutdown task has no clear owner yet'
    if row['shutdown_check_status'] != 'complete':
        return 'complete_shutdown_check', 'shutdown checklist is not complete yet'
    if row['archive_snapshot_status'] != 'complete':
        return 'archive_before_shutdown', 'archive snapshot or final version capture is incomplete'
    if row['next_cohort_seed_status'] != 'seeded':
        return 'prepare_next_cohort_seed', 'next cohort still has no usable warm-start seed'
    return 'shutdown_ready', 'item has an owner, passed shutdown check, has an archive snapshot, and leaves a usable seed'


routed_rows = []
for row in rows:
    route, reason = route_shutdown_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Shutdown workflow routes:')
for row in routed_rows:
    print(
        f"  {row['shutdown_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'shutdown_ready']
check_queue = [row for row in routed_rows if row['route'] == 'complete_shutdown_check']
archive_queue = [row for row in routed_rows if row['route'] == 'archive_before_shutdown']
seed_queue = [row for row in routed_rows if row['route'] == 'prepare_next_cohort_seed']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_shutdown_owner']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'shutdown_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Shutdown-ready queue:', [row['shutdown_id'] for row in ready_queue])
print('Complete-check queue:', [row['shutdown_id'] for row in check_queue])
print('Archive-first queue:', [row['shutdown_id'] for row in archive_queue])
print('Prepare-seed queue:', [row['shutdown_id'] for row in seed_queue])
print('Assign-owner queue:', [row['shutdown_id'] for row in owner_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured shutdown precision:', round(ready_precision, 3))


In [ ]:
def shutdown_steps(row):
    steps = []
    if row['owner_status'] != 'assigned':
        steps.append('先指定收尾 owner，并明确谁归档、谁写 retrospective、谁留 warm-start 种子。')
    if row['shutdown_check_status'] != 'complete':
        steps.append('补齐学期收尾检查：最终版、评分边界、反馈回收和关闭动作。')
    if row['archive_snapshot_status'] != 'complete':
        steps.append('补完整归档快照：关键 notebook、数据、版本和可复现说明。')
    if row['next_cohort_seed_status'] != 'seeded':
        steps.append('为下一届准备种子包：starter repo、kickoff note、最小 handout 和 calendar seed。')
    return steps or ['可以进入 end-of-term shutdown package；保留最终版本号和下一届 warm-start 指针。']


for row in routed_rows:
    if row['route'] != 'shutdown_ready':
        print(f"\n{row['shutdown_id']} -> {row['route']} ({row['shutdown_area']})")
        for step in shutdown_steps(row):
            print(' -', step)


In [ ]:
shutdown_outline = [
    'Final delivery: which files, grades, and notes are the authoritative final version',
    'Feedback digest: student, TA, and instructor lessons worth carrying forward',
    'Archive snapshot: notebooks, data, versions, and reproducibility pointers',
    'Retrospective note: what slowed the course down and what should change next time',
    'Warm-start seed: starter repo, kickoff note, calendar seed, and first-week package',
    'Owner and dates: who shuts down, who archives, who hands the next cohort the first key',
]

warmstart_assistant_prompt = '''你是我的 capstone 收尾与 warm-start 助手。

任务：
1. 先阅读 shutdown-warmstart table，不要只按 release pressure 排序；
2. 先检查 owner；
3. 再检查 shutdown check、archive snapshot 和 next-cohort seed；
4. 把每个模块 route 到 shutdown_ready、complete_shutdown_check、
   archive_before_shutdown、prepare_next_cohort_seed 或 assign_shutdown_owner；
5. 对每个非 ready 模块输出收尾前 checklist。

输出格式：
- Shutdown-ready modules
- Complete-check modules
- Archive-first modules
- Prepare-seed modules
- Assign-owner modules
- Shutdown checklist by module
'''

print('Shutdown package outline:')
for item in shutdown_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(warmstart_assistant_prompt)


In [ ]:
shutdown_snapshot = {
    'dataset': DATA_PATH.name,
    'n_shutdown_items': len(rows),
    'baseline_top4_shutdown_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'shutdown_ready': [row['shutdown_id'] for row in ready_queue],
    'complete_shutdown_check': [row['shutdown_id'] for row in check_queue],
    'archive_before_shutdown': [row['shutdown_id'] for row in archive_queue],
    'prepare_next_cohort_seed': [row['shutdown_id'] for row in seed_queue],
    'assign_shutdown_owner': [row['shutdown_id'] for row in owner_queue],
    'python_version': platform.python_version(),
}

print('Shutdown warm-start snapshot:')
for key, value in shutdown_snapshot.items():
    print(f'  {key}: {value}')
